[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/quadrics/csg_raytracer/csg_raytracer.ipynb)

# Solids Cut from Quadrics, Ray Traced

This example ray-traces constructive solid geometry built from quadric surfaces in projective three-dimensional space. A ruled hyperboloid capped by two horizontal planes, a hollow bowl formed by subtracting a concentric sphere and slicing off the top, and a mirror sphere stand on a checkered floor under a directional lamp while a camera circles them. Every curved surface, cutting plane, and floor half-space is modeled as the same kind of solid quadric, so a single intersection loop traces the entire scene.

Each solid quadric is a symmetric duality map from points to planes (`Plane+ <- Point-`) whose regressive pairing `quadric(point) & point` vanishes on the boundary and is negative inside, while `quadric(point)` gives the outward tangent plane at any surface hit. Symmetrizing a bounding plane with the plane at infinity `w` embeds its negative half-space as `(plane * (w & Point) + w * (plane & Point)) / 2`, negating a quadric swaps inside and outside to subtract an interior cavity, and rigid placement conjugates the map by one motor as `motor >> quadric(motor << Point)`. Inside the WebGPU shader, those same geometric-algebra pairings and tangent-plane reflections `normal >> direction` evaluate ray intersections, lamp cosines, shadows, and mirror bounces directly on typed extensors.

In homogeneous $4 \times 4$ matrix notation, a quadric solid is stored as a symmetric matrix $Q$ evaluated via $\mathbf{p}^\top Q \mathbf{p} \le 0$, rigid placement by $(R, \mathbf{t}) \in \mathrm{SE}(3)$ conjugates $Q$ with the inverse-transpose block matrix $M^{-\top} Q M^{-1}$ where $M = \begin{bmatrix} R & \mathbf{t} \\ \mathbf{0}^\top & 1 \end{bmatrix}$, surface normals drop the homogeneous fourth row of $Q\mathbf{p}$ before Euclidean normalization, and mirror reflection expands into $\mathbf{d} - 2(\hat{\mathbf{n}} \cdot \mathbf{d})\hat{\mathbf{n}}$ alongside a separate linear ray-plane branch $\hat{\mathbf{n}} \cdot (\mathbf{o} + t\mathbf{d}) + d = 0$ for half-spaces.

In [ ]:
# In Colab, install numga and its examples from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np

from numga import NumpyContext, concatenate, stack
from numga.algebras import PGA3D as ga
from numga.wgsl import Shader, supersampled
from examples.quadrics.csg_raytracer import render, scenarios
from examples.util.animation import save_figure, show_states
from examples.util.panels import image_figure

mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                                    # Scalar+
# Points and directions are antivectors, planes are vectors, and a direction is a point at infinity.
Point = ga.gatype.antivector()                                                 # Antivector-
Plane = ga.gatype.vector()                                                     # Vector+
Motor = ga.gatype.rotor()                                                      # Rotor+
Quadric = ga.gatype((Plane, Point))                                            # Plane+ <- Point-
origin = mv.point([0, 0, 0])                                                   # [] Point-
# Coordinate planes through the origin orthogonal to the three spatial axes.
axes = stack((mv.x, mv.y, mv.z))                                               # [3] Plane+

## Ruled hyperboloid and straight-line generators

Subtracting the axial coordinate-plane pairing from an ellipsoid's transverse pairings turns the closed surface into a hyperboloid of one sheet. Through every point on its circular waist pass two straight lines that stay entirely on the surface, so rotating one waist point and both ruling directions around the vertical axis generates both ruling families.

In [ ]:
def ellipsoid(semi_axes: np.ndarray) -> Quadric:
    """An origin-centred solid ellipsoid scales each coordinate-plane pairing inversely by the
    squared `semi_axes` (`[3]`) and subtracts the ideal-plane pairing (`[] Plane+ <- Point-`)."""
    return (axes * (axes & Point) / semi_axes**2).sum(axis=-1) - mv.w * (mv.w & Point)


def hyperboloid(semi_axes: np.ndarray) -> Quadric:
    """An origin-centred ruled hyperboloid of one sheet around the vertical axis subtracts the axial
    pairing from the transverse coordinate-plane pairings (`[] Plane+ <- Point-`)."""
    signs = np.array([1, 1, -1])
    return (axes * (axes & Point) * signs / semi_axes**2).sum(axis=-1) - mv.w * (mv.w & Point)

In [ ]:
def ruling_lines(semi_axes: np.ndarray, n_rulings: int) -> tuple[Point, Point, Point]:
    """Rotating one waist point and its two surface-tangent directions evenly around the vertical
    axis generates both straight-line ruling families (`[n_rulings] Point-`) of a circular
    hyperboloid."""
    radius, _, height = semi_axes
    angles = np.linspace(0, 2 * np.pi, n_rulings, endpoint=False)
    turns = (mv.xy * (-angles / 2)).exp()                                      # [n_rulings] Motor+
    waist = turns >> mv.point([radius, 0, 0])                                  # [n_rulings] Point-
    rising = turns >> mv.direction([0, radius, height])                        # [n_rulings] Point-
    falling = turns >> mv.direction([0, radius, -height])                      # [n_rulings] Point-
    return waist, rising, falling


waist_axes = scenarios.waist_axes
n_rulings = scenarios.n_rulings
n_offsets = 5
offsets = np.linspace(-2, 2, n_offsets)[:, None]

waist, rising, falling = ruling_lines(waist_axes, n_rulings)                   # [n_rulings] Point-, Point-, Point-
# Sample points along both ruling families and pair them against the hyperboloid quadric.
ruling_points = waist + stack([rising, falling])[:, None] * offsets            # [2, n_offsets, n_rulings] Point-
ruling_residuals = hyperboloid(waist_axes)(ruling_points) & ruling_points      # [2, n_offsets, n_rulings] Scalar+

## Composite solids from quadric intersections

Each solid is the intersection of a contiguous slice of quadrics, placed in the scene by pulling input points back and pushing tangent planes forward with one rigid motor `motor >> quadric(motor << Point)`. Alongside its quadric range, each solid carries an RGB colour, a mirror reflectivity, and two planes whose integer steps tile a checker pattern across its surface.

In [ ]:
def half_space(plane: Plane) -> Quadric:
    """Symmetrizing `plane` with the plane at infinity embeds `plane`'s negative half-space as a
    rank-two solid quadric (`[] Plane+ <- Point-`)."""
    return (plane * (mv.w & Point) + mv.w * (plane & Point)) / 2


# Rigid motors translate each solid from the origin along its placement direction after tilting.
hyperboloid_motor = (((mv.w ^ scenarios.hyperboloid_place.dual()) / 2).exp()
                     * (mv.xz * (-scenarios.hyperboloid_tilt / 2)).exp())      # [] Motor+
bowl_shift = ((mv.w ^ scenarios.bowl_place.dual()) / 2).exp()                  # [] Motor+
bowl_tilt = ((mv.yz * (-scenarios.bowl_tilts[0] / 2)).exp()
             * (mv.xz * (-scenarios.bowl_tilts[1] / 2)).exp())                 # [] Motor+
bowl_motor = bowl_shift * bowl_tilt                                            # [] Motor+
mirror_motor = ((mv.w ^ scenarios.mirror_place.dual()) / 2).exp()              # [] Motor+
height = waist_axes[2]

# Cap the ruled hyperboloid between horizontal half-spaces one half-height above and below its waist.
hyperboloid_parts = stack([hyperboloid(waist_axes), half_space(mv.z - mv.w * height),
                           half_space(-mv.z - mv.w * height)])                 # [3] Plane+ <- Point-
# Subtract the inner sphere by negation and slice off the top with the tilted half-space.
bowl_parts = stack([ellipsoid(scenarios.bowl_axes),
                    -ellipsoid(scenarios.bowl_axes * scenarios.wall_fraction),
                    half_space(scenarios.bowl_cut.normalized())])              # [3] Plane+ <- Point-
mirror_parts = ellipsoid(np.full(3, scenarios.mirror_radius)).reshape(1)       # [1] Plane+ <- Point-
floor_parts = half_space(mv.z).cast(Quadric).reshape(1)                        # [1] Plane+ <- Point-

# Pull back each solid's input points and push forward its output planes by the solid's motor.
placed_hyperboloid = hyperboloid_motor >> hyperboloid_parts(hyperboloid_motor << Point)
placed_bowl = bowl_motor >> bowl_parts(bowl_motor << Point)
placed_mirror = mirror_motor >> mirror_parts(mirror_motor << Point)
shapes = [placed_hyperboloid, placed_bowl, placed_mirror, floor_parts]
counts = np.array([shape.shape[0] for shape in shapes])
ends = np.cumsum(counts)
quadrics = concatenate(shapes)                                                 # [8] Plane+ <- Point-
first_stripe = stack([hyperboloid_motor >> ((mv.x + mv.z) * scenarios.stripe_rate),
                      0 * mv.x, 0 * mv.x, mv.x * scenarios.tile_rate])         # [4] Plane+
second_stripe = stack([hyperboloid_motor >> ((mv.y - mv.z) * scenarios.stripe_rate),
                       0 * mv.y, 0 * mv.y, mv.y * scenarios.tile_rate])        # [4] Plane+
solids = dict(first=ends - counts, end=ends, colour=scenarios.colours,
              reflectivity=scenarios.reflectivities, first_stripe=first_stripe,
              second_stripe=second_stripe)

## Ray-quadric intersection and shading in WGSL

Along a camera ray from `origin` along `direction`, pairing a quadric with the moving point yields a scalar quadratic in distance whose coefficients come directly from the quadric's pairings on `origin` and `direction`. A half-space has a vanishing quadratic coefficient, and the cancellation-free root formula evaluates its single plane crossing without a separate branch. At the nearest root that lies inside every sibling quadric of its solid, `quadric(point)` supplies the outward tangent plane, `normal & lamp` gives the diffuse cosine, and `normal >> direction` reflects the viewing ray for both specular glint and mirror reflection.

In [ ]:
source = supersampled() + '''
// Each solid intersects `quadrics[first]` up to `quadrics[end]`, carries its RGB colour and mirror
// reflectivity, and tiles a checker pattern from unit steps of `first_stripe` and `second_stripe`.
struct Solid {
    first: u32,
    end: u32,
    colour: vec3<f32>,
    reflectivity: f32,
    first_stripe: Plane,
    second_stripe: Plane,
}

var<storage, read> solids: array<Solid>;
var<storage, read> quadrics: array<Quadric>;
var<uniform> camera: Motor;
var<uniform> eye: Point;
var<uniform> lamp: Point;
var<uniform> half_view: f32;
alias Sample = vec3<f32>;

// Sentinel distance beyond the scene and ray offset off a hit surface before secondary rays start.
const scene_far = 1e6;
const clearance = 1e-3;
// Inside-test slack for boundary points shared by two intersecting quadric surfaces.
const tolerance = 1e-4;
const ambient = 0.28;
const shadow = 0.3;
const checker = 0.15;
const shininess = 40.0;
const gloss = 0.3;
const horizon = vec3<f32>(0.80, 0.84, 0.90);
const zenith = vec3<f32>(0.38, 0.52, 0.72);

struct Hit {
    distance: f32,
    tangent: Plane,
    solid: u32,
}

// Test whether `point` lies inside every quadric of `solid` other than `own`.
fn inside_others(solid: Solid, own: u32, point: Point) -> bool {
    for (var index = solid.first; index < solid.end; index++) {
        if (index != own && (quadrics[index](point) & point) > tolerance) {
            return false;
        }
    }
    return true;
}

// Nearest boundary hit across all solids, lying on one quadric and inside all sibling quadrics.
fn first_hit(origin: Point, direction: Point) -> Hit {
    var hit = Hit(scene_far, mv.z, 0u);
    for (var index = 0u; index < arrayLength(&solids); index++) {
        let solid = solids[index];
        for (var own = solid.first; own < solid.end; own++) {
            let quadric = quadrics[own];
            // Quadratic coefficients a, 2b, and c of the quadric pairing along `origin + direction * distance`.
            let a = quadric(direction) & direction;
            let b = ((quadric(direction) & origin) + (quadric(origin) & direction)) / 2;
            let c = quadric(origin) & origin;
            let discriminant = b * b - a * c;
            if (discriminant < 0) {
                continue;
            }
            // Cancellation-free roots; a half-space has `a == 0` and takes only the `c / q` root.
            let q = -(b + select(-1.0, 1.0, b >= 0) * sqrt(discriminant));
            let roots = vec2<f32>(select(q / a, -1.0, a == 0), c / q);
            for (var which = 0u; which < 2u; which++) {
                let distance = roots[which];
                let point = origin + direction * distance;
                if (distance > clearance && distance < hit.distance && inside_others(solid, own, point)) {
                    hit = Hit(distance, quadric(point), index);
                }
            }
        }
    }
    return hit;
}

struct Sight {
    colour: vec3<f32>,
    point: Point,
    normal: Plane,
    reflectivity: f32,
}

// Direct illumination, shadow test, planar checker, and specular glint at a ray's first hit, or sky.
fn sight(origin: Point, direction: Point) -> Sight {
    let hit = first_hit(origin, direction);
    if (hit.distance >= scene_far) {
        return Sight(mix(horizon, zenith, clamp(mv.z & direction, 0.0, 1.0)), origin, mv.z, 0.0);
    }
    let solid = solids[hit.solid];
    let point = origin + direction * hit.distance;
    let normal = hit.tangent.normalized();
    // Pairing the unit tangent plane with the unit lamp direction yields the incidence cosine.
    let lit = max(normal & lamp, 0.0) * select(1.0, shadow, first_hit(point + lamp * clearance, lamp).distance < scene_far);
    let parity = abs((floor(solid.first_stripe & point) + floor(solid.second_stripe & point)) % 2.0);
    // Reflect the view direction in the tangent plane and pair its dual plane against the lamp direction.
    let glint = pow(max(-((normal >> direction).dual() & lamp), 0.0), shininess);
    let colour = solid.colour * (1 - checker * parity) * (ambient + (1 - ambient) * lit) + gloss * glint * lit;
    return Sight(colour, point, normal, solid.reflectivity);
}

// Trace one supersampled screen ray along the camera's local -z axis and blend one mirror reflection.
fn sample(place: vec2<f32>) -> Sample {
    let offset = screen(place) * half_view;
    let toward = camera >> (-mv.z.dual() + mv.x.dual() * offset.x + mv.y.dual() * offset.y);
    let direction = toward / toward.dual().norm();
    let seen = sight(eye, direction);
    // Sandwiching the ray direction by the unit tangent plane reflects the view ray across the surface.
    let reflected = seen.normal >> direction;
    let mirrored = sight(seen.point + reflected * clearance, reflected);
    return mix(seen.colour, mirrored.colour, seen.reflectivity);
}
'''

## Camera pose motor and front view

A camera starts at the origin aiming along its local negative vertical-depth axis, backs off by `distance`, pitches down from `elevation`, turns in azimuth, and translates to `target`. `supersampled()` averages a rotated two-by-two subpixel grid inside each pixel by calling `sample`.

In [ ]:
def camera_orbit(target: Point, distance: float, elevation: float, azimuth: np.ndarray) -> Motor:
    """Backing a camera off along its viewing axis by `distance`, pitching to `elevation`, turning
    by `azimuth`, and translating to `target` forms its pose motor (`[...] Motor+`)."""
    back_off = (mv.zw * (distance / 2)).exp()                                  # [] Motor+
    pitch = (mv.yz * (-(np.pi / 2 - elevation) / 2)).exp()                     # [] Motor+
    turn = (mv.xy * (-azimuth / 2)).exp()                                      # [...] Motor+
    # Translate from the origin to `target` along the ideal point joining their difference.
    translation = ((mv.w ^ (target - origin).dual()) / 2).exp()                # [] Motor+
    return translation * turn * pitch * back_off


def orbit_cameras(n_cameras: int) -> Iterator[Motor]:
    """Yields `n_cameras` camera pose motors (`[] Motor+`) spaced evenly in azimuth around `target`."""
    for azimuth in np.linspace(0, 2 * np.pi, n_cameras, endpoint=False):
        yield camera_orbit(scenarios.target, scenarios.distance, scenarios.elevation, azimuth)


tracer = Shader(source, ga, dict(Point=Point, Plane=Plane, Motor=Motor, Quadric=Quadric), {})
still, = render.frames(tracer, quadrics, solids, orbit_cameras(1), scenarios.lamp,
                       scenarios.half_view, scenarios.image_shape)
save_figure(image_figure(still), "csg_raytracer");

On the left, diagonal checker planes trace curved bands across the tilted orange hyperboloid between its flat end caps; on the right, the tilted cutting plane exposes the teal bowl's spherical cavity and wall thickness, while the central mirror sphere reflects both solids, the checkered floor, and their hard lamp shadows.

## Orbiting the scene

Streaming camera motors around a full revolution traces each viewpoint on the GPU in a single pass while recording the pose motors for verification.

In [ ]:
n_frames = scenarios.n_frames
frame_duration_ms = scenarios.frame_duration_ms


def orbit_frames(cameras: Iterator[Motor]) -> Iterator[np.ndarray]:
    """Yields `[rows, columns, 3]` RGB frames traced for each streamed camera motor."""
    return render.frames(tracer, quadrics, solids, cameras, scenarios.lamp,
                         scenarios.half_view, scenarios.animation_shape)


streamed_cameras = stack(show_states(
    orbit_cameras(n_frames), orbit_frames,
    name="csg_raytracer", duration_ms=frame_duration_ms, n_frames=n_frames,
))                                                                             # [n_frames] Motor+

As the camera circles the target, the mirror sphere sweeps reflections of the hyperboloid and hollow bowl across its surface while the lamp's cast shadows stay anchored to the floor tiles.

In [ ]:
# checks
# Both straight-line ruling families lie on the hyperboloid surface, and every streamed camera aims at `target`.
assert (ruling_residuals.abs() < 1e-10).all()
eyes = streamed_cameras >> origin                                              # [n_frames] Point-
aim = streamed_cameras >> (-mv.z.dual())                                       # [n_frames] Point-
assert ((eyes + aim * scenarios.distance - scenarios.target).antinorm() < 1e-10).all()